# 06. 폭염·폭우가 업종별 매출에 주는 영향

**읽는 것**: `data/processed/analysis_visitor.csv` (04 실행 결과), `data/raw/card/데이터2.txt` (원업종·연령 분석용), `docs/industry_mapping.csv`

**기간**: 2025년 7~9월, 공휴일(8/15) 제외

**기본 모형**: `log(매출) ~ 폭염 + 비 + 호우 + 주차 + 요일`
- 주차: 같은 주끼리 비교해서 휴가철 효과 제거
- 요일: 요일 효과 제거
- 폭염 = 일최고기온 33℃ 이상, 비 = 1mm 이상, 호우 = 30mm 이상 (호우는 비 효과에 **추가로** 더해지는 효과)
- `*` = p < 0.05, 견고한 표준오차(HC1)

**구성**

| 섹션 | 질문 | 저장 파일 |
|---|---|---|
| A | 업종 × 지역별로 폭염·폭우가 매출을 얼마나 바꾸나 | 06_A_effects |
| B | 같은 날씨에서 춘천이 강남보다 유의하게 더 변하나 | 06_B_region_diff |
| C | 빠지는 건 외지인 소비인가 지역민 소비인가 | 06_C_visitor |
| D | 원업종 단위로 보면 무엇이 움직였나 | 06_D_industry |
| E | 강수량·기온 구간별로 단계적으로 변하나 (지원 기준) | 06_E_rain_dose, 06_E_temp_dose |
| F | 뚜렷한 임계값이 있나 (참고) | 06_F_threshold |
| G | 기준을 바꿔도 결과가 유지되나 | 06_G_robustness |
| H | 연령대별로 다르게 반응하나 | 06_H2~H4 |

표는 `outputs/tables/`, 그래프는 `outputs/figures/`에 저장됩니다.

## 0. 설정 및 데이터

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

BASE = Path("/content/drive/MyDrive/bigcontest-2026")
OUT = BASE / "data/processed"
RAW_CARD = BASE / "data/raw/card"
DOCS = BASE / "docs"
FIG = BASE / "outputs/figures"
TAB = BASE / "outputs/tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

In [ ]:
!apt-get -qq install -y fonts-nanum > /dev/null
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
av = pd.read_csv(OUT / "analysis_visitor.csv", parse_dates=["date"])
av = av[(av["summer"] == 1) & (av["holiday"] == 0)].copy()   # 7~9월, 광복절 제외

DAY_COLS = ["date", "region", "dow", "week", "heat", "tmax", "feels_max", "rain", "heat_streak"]

def add_weather_flags(df, heat_c=33, heavy_mm=30, use_feels=False):
    df = df.copy()
    df["heat"] = ((df["feels_max"] if use_feels else df["tmax"]) >= heat_c).astype(int)
    df["rainy"] = (df["rain"] >= 1).astype(int)
    df["heavy"] = (df["rain"] >= heavy_mm).astype(int)
    df["lsales"] = np.log(df["sales"])
    return df

# 업종 단위 (외지인 + 지역민 합산)
grp = av.groupby(DAY_COLS + ["ind_grp"])["sales"].sum().reset_index()
grp = add_weather_flags(grp)
print(grp.shape)
print(grp.drop_duplicates(["date", "region"]).groupby("region")[["heat", "rainy", "heavy"]].sum())

## A. 업종 × 지역 날씨 영향

- `폭염`: 폭염일 매출 변화
- `비`: 비 온 날(1~30mm) 매출 변화
- `호우일 전체`: 비 + 호우 추가 효과를 합친 값

In [ ]:
FORMULA = "lsales ~ heat + rainy + heavy + C(week) + C(dow)"

def fit_effects(df, keys, formula=FORMULA):
    rows = []
    for k, d in df.groupby(keys):
        if d["heat"].sum() < 3 or len(d) < 40:
            continue
        m = smf.ols(formula, data=d).fit(cov_type="HC1")
        storm = m.t_test("rainy + heavy = 0")
        rows.append(dict(zip(keys, k if isinstance(k, tuple) else (k,)),
                         heat=np.exp(m.params["heat"]) - 1, heat_p=m.pvalues["heat"],
                         rainy=np.exp(m.params["rainy"]) - 1, rainy_p=m.pvalues["rainy"],
                         storm=np.exp(float(storm.effect[0])) - 1, storm_p=float(storm.pvalue),
                         n_days=len(d)))
    return pd.DataFrame(rows)

def show(df, index, cols, labels):
    out = pd.DataFrame(index=df.set_index(index).index)
    for c, lab in zip(cols, labels):
        v = df.set_index(index)
        out[lab] = v[c].map("{:+.1%}".format) + v[c + "_p"].map(lambda p: "*" if p < 0.05 else "")
    return out

eff = fit_effects(grp, ["region", "ind_grp"])
eff.to_csv(TAB / "06_A_effects.csv", index=False, encoding="utf-8-sig")
print(show(eff, ["ind_grp", "region"], ["heat", "rainy", "storm"], ["폭염", "비", "호우일 전체"])
      .unstack("region").to_string())

In [ ]:
order = eff.pivot(index="ind_grp", columns="region", values="storm")["춘천"].sort_values().index

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
for ax, (col, title) in zip(axes, [("heat", "폭염일"), ("storm", "호우일")]):
    t = eff.pivot(index="ind_grp", columns="region", values=col).loc[order]   # 두 그래프 같은 순서
    t.plot.barh(ax=ax, color={"강남": "tab:blue", "춘천": "tab:orange"}, legend=(col == "heat"))
    ax.axvline(0, color="black", lw=0.8)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:+.0%}"))
    ax.set_title(f"{title} 매출 변화 (주차·요일·강수 통제)")
    ax.set_ylabel("")
plt.tight_layout()
plt.savefig(FIG / "06_A_effects.png", dpi=150)
plt.show()

## B. 강남 vs 춘천 차이 검정

두 지역을 한 모형에 넣고 `날씨 × 춘천` 상호작용항을 봅니다. 주차·요일 효과는 지역별로 따로 둡니다.

- −20%p면 같은 날씨에서 춘천이 강남보다 20%p 더 감소
- `*`면 두 지역 차이가 통계적으로 유의

In [ ]:
grp["chun"] = (grp["region"] == "춘천").astype(int)
F_DIFF = ("lsales ~ heat + rainy + heavy + chun:heat + chun:rainy + chun:heavy"
          " + C(region):C(week) + C(region):C(dow)")

rows = []
for g, d in grp.groupby("ind_grp"):
    m = smf.ols(F_DIFF, data=d).fit(cov_type="HC1")
    storm = m.t_test("chun:rainy + chun:heavy = 0")
    rows.append({"ind_grp": g,
                 "폭염 차이": m.params["chun:heat"], "폭염 p": m.pvalues["chun:heat"],
                 "호우일 차이": float(storm.effect[0]), "호우일 p": float(storm.pvalue)})
diff = pd.DataFrame(rows).set_index("ind_grp")
diff.to_csv(TAB / "06_B_region_diff.csv", encoding="utf-8-sig")

shown = pd.DataFrame({
    "폭염: 춘천−강남": (diff["폭염 차이"] * 100).map("{:+.1f}%p".format) + diff["폭염 p"].map(lambda p: "*" if p < 0.05 else ""),
    "호우일: 춘천−강남": (diff["호우일 차이"] * 100).map("{:+.1f}%p".format) + diff["호우일 p"].map(lambda p: "*" if p < 0.05 else ""),
})
print("같은 날씨에서 춘천이 강남보다 얼마나 더 변하나 (로그 차이 ≈ %p, * = p<0.05)")
print(shown.to_string())

## C. 외지인 vs 지역민

외지인 = 고객 거주 시도 ≠ 가맹점 시도 (강남=서울, 춘천=강원)

In [ ]:
vis = av.groupby(DAY_COLS + ["ind_grp", "is_visitor"])["sales"].sum().reset_index()
vis = add_weather_flags(vis)
vis["who"] = vis["is_visitor"].map({1: "외지인", 0: "지역민"})

eff_v = fit_effects(vis, ["region", "ind_grp", "who"])
eff_v.to_csv(TAB / "06_C_visitor.csv", index=False, encoding="utf-8-sig")

for r in ["춘천", "강남"]:
    print(f"\n[{r}] 외지인 vs 지역민")
    print(show(eff_v[eff_v["region"] == r], ["ind_grp", "who"], ["heat", "storm"], ["폭염", "호우일"])
          .unstack("who").to_string())

## D. 원업종 단위

### D1. 강남: 폭염 때 무엇이 늘고 무엇이 줄었나

지역 매출 비중 0.1% 미만 업종은 값이 튀어서 제외합니다. 원업종 약 50개를 검정하므로 2~3개는 우연히 유의할 수 있습니다.

In [ ]:
mapping = pd.read_csv(DOCS / "industry_mapping.csv")
keep = mapping[~mapping["ind_grp"].str.startswith("제외")]

raw = pd.read_csv(RAW_CARD / "신한카드_빅콘테스트2026_데이터2.txt", sep="\t", encoding="cp949",
                  dtype=str, usecols=["TA_YMD", "MCT_SGG_CD", "MCT_RY_CD", "TS_AT"])
raw = raw.rename(columns={"TA_YMD": "date", "MCT_SGG_CD": "region",
                          "MCT_RY_CD": "industry", "TS_AT": "sales"})
raw["date"] = pd.to_datetime(raw["date"], format="%Y%m%d")
raw["region"] = raw["region"].map({"서울 강남구": "강남", "강원 춘천시": "춘천"})
raw["sales"] = pd.to_numeric(raw["sales"])
raw = raw[raw["industry"].isin(keep["industry"])]

ind = raw.groupby(["date", "region", "industry"])["sales"].sum().reset_index()
days = grp[DAY_COLS].drop_duplicates()
ind = add_weather_flags(ind.merge(days, on=["date", "region"]))     # 여름·비공휴일만 남음
share = ind.groupby(["region", "industry"])["sales"].sum()
share = share / share.groupby(level="region").transform("sum")
ind = ind.merge(share.rename("share").reset_index(), on=["region", "industry"])
ind = ind[(ind["share"] >= 0.001)]

eff_i = fit_effects(ind, ["region", "industry", "share"])
eff_i = eff_i.merge(keep, on="industry", how="left")
eff_i.to_csv(TAB / "06_D_industry.csv", index=False, encoding="utf-8-sig")

g = eff_i[eff_i["region"] == "강남"].sort_values("heat")
print("[강남] 원업종별 폭염 효과 (매출 비중 0.1% 이상)")
print(g.assign(share=g["share"].map("{:.1%}".format),
               heat=g["heat"].map("{:+.1%}".format) + g["heat_p"].map(lambda p: " *" if p < 0.05 else ""))
       [["industry", "ind_grp", "share", "heat"]].to_string(index=False))

### D2. 춘천 실내여가: 감소를 주도한 업종

In [ ]:
c = eff_i[(eff_i["region"] == "춘천") & (eff_i["ind_grp"] == "실내여가")].sort_values("share", ascending=False)
print("[춘천] 실내여가 원업종별 폭염 효과")
print(c.assign(share=c["share"].map("{:.2%}".format),
               heat=c["heat"].map("{:+.1%}".format) + c["heat_p"].map(lambda p: " *" if p < 0.05 else ""))
       [["industry", "share", "heat"]].to_string(index=False))

## E. 강수량·기온 구간별 매출 변화 (지원 기준)

한 점짜리 임계값 대신, 구간을 나눠 **단계적으로 변하는지** 봅니다. 구간이 커질수록 순서대로 커지면 우연으로 보기 어렵습니다.

- 강수: 무강수(기준) / 1~10mm / 10~30mm / 30mm+
- 기온: ~28℃ / 28~31℃(기준) / 31~33℃ / 33~35℃ / 35℃~

대상은 `RAIN_DOSE_TARGETS`, `TEMP_DOSE_TARGETS`에서 바꿀 수 있습니다.

In [ ]:
rain_bins = [-0.1, 0.9, 10, 30, 1000]
rain_labels = ["무강수", "1~10mm", "10~30mm", "30mm+"]
temp_bins = [0, 28, 31, 33, 35, 50]
temp_labels = ["~28℃", "28~31℃", "31~33℃", "33~35℃", "35℃~"]

def dose(region, g, kind):
    d = grp[(grp["region"] == region) & (grp["ind_grp"] == g)].copy()
    if kind == "rain":
        d["bin"] = pd.cut(d["rain"], rain_bins, labels=rain_labels)
        base, labels = "무강수", rain_labels
        f = "lsales ~ C(bin, Treatment('무강수')) + heat + C(week) + C(dow)"
    else:
        d["bin"] = pd.cut(d["tmax"], temp_bins, labels=temp_labels)
        base, labels = "28~31℃", temp_labels
        f = "lsales ~ C(bin, Treatment('28~31℃')) + rainy + heavy + C(week) + C(dow)"
    m = smf.ols(f, data=d).fit(cov_type="HC1")
    ci = m.conf_int()
    rows = []
    for lab in labels:
        n = int((d["bin"] == lab).sum())
        if lab == base:
            rows.append({"대상": f"{region} {g}", "구간": lab, "일수": n,
                         "변화": 0.0, "하한": 0.0, "상한": 0.0, "p": np.nan})
            continue
        name = [p for p in m.params.index if f"[T.{lab}]" in p]
        if not name:
            continue
        b, (lo, hi) = m.params[name[0]], ci.loc[name[0]]
        rows.append({"대상": f"{region} {g}", "구간": lab, "일수": n,
                     "변화": np.exp(b) - 1, "하한": np.exp(lo) - 1, "상한": np.exp(hi) - 1,
                     "p": m.pvalues[name[0]]})
    return rows

RAIN_DOSE_TARGETS = [("춘천", "야외레저"), ("춘천", "카페·제과"), ("춘천", "음식점"), ("강남", "야외레저")]
TEMP_DOSE_TARGETS = [("춘천", "실내여가"), ("강남", "대형유통")]
rain_dose = pd.DataFrame(sum([dose(r, g, "rain") for r, g in RAIN_DOSE_TARGETS], []))
temp_dose = pd.DataFrame(sum([dose(r, g, "temp") for r, g in TEMP_DOSE_TARGETS], []))
rain_dose.to_csv(TAB / "06_E_rain_dose.csv", index=False, encoding="utf-8-sig")
temp_dose.to_csv(TAB / "06_E_temp_dose.csv", index=False, encoding="utf-8-sig")

def fmt(df):
    return df.assign(**{"매출 변화": df["변화"].map("{:+.1%}".format) + df["p"].map(lambda p: " *" if p < 0.05 else ""),
                        "95% 구간": df["하한"].map("{:+.0%}".format) + " ~ " + df["상한"].map("{:+.0%}".format)}
                     )[["대상", "구간", "일수", "매출 변화", "95% 구간"]]
print("[강수 구간별]  기준 = 무강수\n", fmt(rain_dose).to_string(index=False))
print("\n[기온 구간별]  기준 = 28~31℃\n", fmt(temp_dose).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, df, order, ref, title in [
    (axes[0], rain_dose, rain_labels, ("30mm+", "호우 30mm"), "강수량 구간별 매출 변화 (기준: 무강수)"),
    (axes[1], temp_dose, temp_labels, ("33~35℃", "폭염 33℃"), "기온 구간별 매출 변화 (기준: 28~31℃)"),
]:
    x = {lab: i for i, lab in enumerate(order)}
    for target, d in df.groupby("대상", sort=False):
        d = d.set_index("구간").loc[[o for o in order if o in d["구간"].values]]
        xs = [x[i] for i in d.index]
        y, lo, hi = d["변화"] * 100, d["하한"] * 100, d["상한"] * 100
        ax.errorbar(xs, y, yerr=[y - lo, hi - y], marker="o", capsize=4, label=target)
    ax.axhline(0, color="gray", lw=0.8)
    ax.axvline(x[ref[0]] - 0.5, color="gray", ls="--", lw=1)
    ax.text(x[ref[0]] - 0.45, ax.get_ylim()[1] * 0.9, ref[1], color="gray", fontsize=9)
    ax.set_xticks(range(len(order)), order)
    ax.set_ylabel("매출 변화 (%)")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.savefig(FIG / "06_E_dose.png", dpi=150)
plt.show()

## F. 임계값 탐색 (참고)

**기온**: k℃를 넘으면 기울기가 바뀐다고 보고 28~36℃에서 0.5℃씩 탐색

**강수량**: k mm 이상이면 한 단계 떨어진다고 보고 5~60mm에서 5mm씩 탐색

**신뢰구간**: 주 단위 블록 부트스트랩 (약 3~5분 소요)

구간이 격자 전체에 걸쳐 넓으면 뚜렷한 꺾임이 없다는 뜻입니다. 이 경우 지원 기준은 E의 구간별 결과를 사용합니다.

In [ ]:
TEMP_TARGETS = [("춘천", "실내여가"), ("춘천", "숙박"), ("춘천", "야외레저"), ("강남", "대형유통")]
RAIN_TARGETS = [("춘천", "야외레저"), ("춘천", "숙박"), ("춘천", "음식점"), ("강남", "음식점")]
TEMP_GRID = np.arange(28, 36.01, 0.5)
RAIN_GRID = np.arange(5, 60.01, 5)
N_BOOT = 500   # 코랩에서 약 3~5분. 시간 여유 있으면 1000

def fe_matrix(d):
    return pd.get_dummies(d[["week", "dow"]].astype(str), drop_first=True).astype(float).values

def best_k(d, kind):
    y = d["lsales"].values
    fe = fe_matrix(d)
    one = np.ones((len(d), 1))
    best = (None, np.inf, None)
    for k in (TEMP_GRID if kind == "temp" else RAIN_GRID):
        if kind == "temp":
            t = d["tmax"].values
            x = np.column_stack([t, np.maximum(t - k, 0), d["rainy"], d["heavy"]])
            if (t > k).sum() < 5:
                continue
        else:
            step = (d["rain"].values >= k).astype(float)
            if step.sum() < 4:
                continue
            x = np.column_stack([d["rain"].values >= 1, step, d["heat"]])
        X = np.column_stack([one, x, fe])
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        ssr = ((y - X @ beta) ** 2).sum()
        if ssr < best[1]:
            effect = beta[2]   # temp: k 이후 추가 기울기 / rain: k mm 이상 추가 효과
            best = (k, ssr, effect)
    return best[0], best[2]

def threshold(region, g, kind, n_boot=N_BOOT, seed=0):
    d = grp[(grp["region"] == region) & (grp["ind_grp"] == g)].reset_index(drop=True)
    k, effect = best_k(d, kind)
    rng = np.random.default_rng(seed)
    weeks = d["week"].unique()
    boots = []
    for _ in range(n_boot):
        pick = rng.choice(weeks, len(weeks), replace=True)
        b = pd.concat([d[d["week"] == w].assign(week=f"{w}_{i}") for i, w in enumerate(pick)])
        kb, _ = best_k(b, kind)
        if kb is not None:
            boots.append(kb)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"region": region, "ind_grp": g, "kind": "기온(℃)" if kind == "temp" else "강수(mm)",
            "임계값": k, "95%_하한": lo, "95%_상한": hi,
            "효과": effect, "효과_설명": "k℃ 초과 1℃당 추가 변화" if kind == "temp" else "k mm 이상일 때 추가 변화"}

res = [threshold(r, g, "temp") for r, g in TEMP_TARGETS] + \
      [threshold(r, g, "rain") for r, g in RAIN_TARGETS]
thr = pd.DataFrame(res)
thr["효과"] = (np.exp(thr["효과"]) - 1).map("{:+.1%}".format)
thr.to_csv(TAB / "06_F_threshold.csv", index=False, encoding="utf-8-sig")
print(thr.to_string(index=False))

## G. 강건성: 기준을 바꿔도 결과가 유지되나

폭염(31℃, 32℃, 체감온도 33℃)과 호우(20mm, 50mm) 정의를 바꿔 핵심 결과를 다시 계산합니다.

In [ ]:
KEY = [("춘천", "야외레저"), ("춘천", "실내여가"), ("춘천", "숙박"),
       ("강남", "대형유통"), ("강남", "음식점"), ("춘천", "음식점")]
SPECS = {
    "기본 (33℃ / 30mm)":     dict(heat_c=33, heavy_mm=30),
    "폭염 31℃":              dict(heat_c=31, heavy_mm=30),
    "폭염 32℃":              dict(heat_c=32, heavy_mm=30),
    "체감온도 33℃":           dict(heat_c=33, heavy_mm=30, use_feels=True),
    "호우 20mm":             dict(heat_c=33, heavy_mm=20),
    "호우 50mm":             dict(heat_c=33, heavy_mm=50),
}
base = grp.drop(columns=["heat", "rainy", "heavy", "lsales"])
rows = []
for name, kw in SPECS.items():
    e = fit_effects(add_weather_flags(base, **kw), ["region", "ind_grp"])
    e = e.set_index(["region", "ind_grp"]).loc[KEY]
    for (r, g), v in e.iterrows():
        rows.append({"기준": name, "대상": f"{r} {g}",
                     "폭염": f"{v['heat']:+.1%}" + ("*" if v["heat_p"] < 0.05 else ""),
                     "호우일": f"{v['storm']:+.1%}" + ("*" if v["storm_p"] < 0.05 else "")})
rob = pd.DataFrame(rows)
rob.to_csv(TAB / "06_G_robustness.csv", index=False, encoding="utf-8-sig")
print(rob.pivot(index="대상", columns="기준", values="폭염")[list(SPECS)].to_string())
print()
print(rob.pivot(index="대상", columns="기준", values="호우일")[list(SPECS)].to_string())

## H. 연령대별 반응

### H1. 연령 데이터 준비

10~30대 / 40~50대 / 60대 이상으로 묶습니다.

In [ ]:
AGE_MAP = {"10대": "10~30대", "20대": "10~30대", "30대": "10~30대",
           "40대": "40~50대", "50대": "40~50대",
           "60대": "60대 이상", "70대": "60대 이상", "80대": "60대 이상", "90대": "60대 이상"}

a = pd.read_csv(RAW_CARD / "신한카드_빅콘테스트2026_데이터2.txt", sep="\t", encoding="cp949",
                dtype=str, usecols=["TA_YMD", "MCT_SGG_CD", "MCT_RY_CD", "AGE_CCD", "TS_AT"])
a = a.rename(columns={"TA_YMD": "date", "MCT_SGG_CD": "region", "MCT_RY_CD": "industry",
                      "AGE_CCD": "age", "TS_AT": "sales"})
a["date"] = pd.to_datetime(a["date"], format="%Y%m%d")
a["region"] = a["region"].map({"서울 강남구": "강남", "강원 춘천시": "춘천"})
a["sales"] = pd.to_numeric(a["sales"])
a["age_grp"] = a["age"].str.replace(" ", "").map(AGE_MAP)
a = a.merge(keep, on="industry")                       # 제외 업종 빠짐
days = grp[DAY_COLS].drop_duplicates()
print(a.shape, a["age_grp"].value_counts().to_dict())

### H2. 춘천 실내여가: 유흥업소 vs 일반 여가 × 연령

유흥업소는 공공 지원 대상에서 보통 제외되는 업종이라 따로 봅니다.

In [ ]:
l = a[(a["region"] == "춘천") & (a["ind_grp"] == "실내여가")].copy()
l["sub"] = np.where(l["industry"] == "유흥업소", "유흥업소", "일반 여가")
l = l.groupby(["date", "region", "sub", "age_grp"])["sales"].sum().reset_index()
l = add_weather_flags(l.merge(days, on=["date", "region"]))
e1 = fit_effects(l, ["region", "sub", "age_grp"])
e1.to_csv(TAB / "06_H2_leisure_age.csv", index=False, encoding="utf-8-sig")
print("[춘천 실내여가] 유흥업소 vs 일반 여가 × 연령")
print(show(e1, ["sub", "age_grp"], ["heat"], ["폭염"]).unstack("age_grp").to_string())

### H3. 업종 × 연령 (탐색용)

78칸을 검정하므로 칸 하나의 `*`보다 **열 전체의 패턴**을 봅니다.

In [ ]:
b = a.groupby(["date", "region", "ind_grp", "age_grp"])["sales"].sum().reset_index()
b = add_weather_flags(b.merge(days, on=["date", "region"]))
e2 = fit_effects(b, ["region", "ind_grp", "age_grp"])
e2.to_csv(TAB / "06_H3_age.csv", index=False, encoding="utf-8-sig")
for r in ["춘천", "강남"]:
    print(f"\n[{r}] 업종 × 연령 폭염 효과")
    print(show(e2[e2["region"] == r], ["ind_grp", "age_grp"], ["heat"], ["폭염"])
          .unstack("age_grp").to_string())

### H4. 여가·외출형 vs 필수형 × 연령 (핵심)

업종을 두 유형으로 묶어 검정 수를 12개로 줄입니다. 유흥업소 제외.

- 여가·외출형: 야외레저, 숙박, 스포츠, 실내여가
- 필수형: 식료품소매, 음식점, 의료, 카페·제과

In [ ]:
LEISURE = ["야외레저", "숙박", "스포츠", "실내여가"]          # 여가·외출형 (유흥업소 제외)
ESSENTIAL = ["식료품소매", "음식점", "의료", "카페·제과"]     # 필수형

a2 = a[a["industry"] != "유흥업소"].copy()
a2["type"] = np.select([a2["ind_grp"].isin(LEISURE), a2["ind_grp"].isin(ESSENTIAL)],
                       ["여가·외출형", "필수형"], default=None)
a2 = a2.dropna(subset=["type"])
t = a2.groupby(["date", "region", "type", "age_grp"])["sales"].sum().reset_index()
t = add_weather_flags(t.merge(days, on=["date", "region"]))
e3 = fit_effects(t, ["region", "type", "age_grp"])
e3.to_csv(TAB / "06_H4_age_type.csv", index=False, encoding="utf-8-sig")
print(show(e3, ["region", "type", "age_grp"], ["heat"], ["폭염"]).unstack("age_grp").to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
ages = ["10~30대", "40~50대", "60대 이상"]
for ax, r in zip(axes, ["강남", "춘천"]):
    d = e3[e3["region"] == r].pivot(index="age_grp", columns="type", values="heat")
    d = d.loc[[x for x in ages if x in d.index]] * 100
    d.plot.bar(ax=ax, color={"여가·외출형": "tab:red", "필수형": "tab:gray"}, rot=0, legend=(r == "강남"))
    sig = e3[e3["region"] == r].set_index(["age_grp", "type"])["heat_p"]
    for i, age in enumerate(d.index):
        for j, typ in enumerate(["여가·외출형", "필수형"]):
            if sig.get((age, typ), 1) < 0.05:
                ax.text(i + (j - 0.5) * 0.25, d.loc[age, typ], "*", ha="center", fontsize=14,
                        va="top" if d.loc[age, typ] < 0 else "bottom")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"{r}: 폭염일 연령별 소비 변화")
    ax.set_xlabel("")
    ax.set_ylabel("매출 변화 (%)")
plt.tight_layout()
plt.savefig(FIG / "06_H4_age_type.png", dpi=150)
plt.show()

In [ ]:
# I1. 핵심 업종 × 시간대별 폭염·호우 효과
at = pd.read_csv(OUT / "analysis_time.csv", parse_dates=["date"])
at = at[(at["summer"] == 1) & (at["holiday"] == 0)].copy()
at = add_weather_flags(at)

KEYS_T = [("강남", "대형유통"), ("강남", "카페·제과"),
          ("춘천", "야외레저"), ("춘천", "음식점"), ("춘천", "카페·제과")]
sub = at[pd.MultiIndex.from_frame(at[["region", "ind_grp"]]).isin(KEYS_T)].copy()

eff_t = fit_effects(sub, ["region", "ind_grp", "time_gb"])
eff_t.to_csv(TAB / "06_I1_timeband.csv", index=False, encoding="utf-8-sig")
print("[시간대별] 폭염·호우일 효과")
print(show(eff_t, ["region", "ind_grp", "time_gb"], ["heat", "storm"], ["폭염", "호우일"])
      .unstack("time_gb").to_string())

In [ ]:
# I2. 그 시간대에 비가 왔을 때 (시간대 강수 매칭)
sub["rain_now"] = (sub["rain_tg"] >= 1).astype(int)

rows = []
for (r, g, tg), d in sub.groupby(["region", "ind_grp", "time_gb"]):
    if d["rain_now"].sum() < 5:
        continue
    m = smf.ols("lsales ~ rain_now + heat + C(week) + C(dow)", data=d).fit(cov_type="HC1")
    rows.append({"region": r, "ind_grp": g, "time_gb": tg,
                 "비 온 시간대 일수": int(d["rain_now"].sum()),
                 "효과": np.exp(m.params["rain_now"]) - 1, "p": m.pvalues["rain_now"]})
rn = pd.DataFrame(rows)
rn.to_csv(TAB / "06_I2_rain_now.csv", index=False, encoding="utf-8-sig")
rn["효과"] = rn["효과"].map("{:+.1%}".format) + rn["p"].map(lambda p: " *" if p < 0.05 else "")
print("[같은 시간대에 비가 왔을 때]")
print(rn.pivot_table(index=["region", "ind_grp"], columns="time_gb", values="효과", aggfunc="first").to_string())

## J. 결과 신뢰성 점검

- J1. 다중 검정 보정 (FDR): 여러 개를 동시에 검정하면 우연히 유의한 게 섞이므로, 보정 후에도 유의한지 확인
- J2. 연속된 날짜 상관 보정 (Newey-West): 하루하루 매출이 서로 비슷해서 표준오차가 작게 잡혔을 수 있으므로 핵심 결과만 다시 확인
- J3. 가짜 폭염 검정 (위약 검정): 폭염이 없던 10~11월에 가짜 폭염일을 무작위로 지정해도 효과가 나오는지 확인

J1. 다중 검정 보정

In [ ]:
from statsmodels.stats.multitest import multipletests

# A 표 전체: 업종 × 지역 × (폭염, 비, 호우일) = 78개 검정을 한 번에 보정
long = eff.melt(id_vars=["region", "ind_grp"], value_vars=["heat_p", "rainy_p", "storm_p"],
                var_name="날씨", value_name="p")
long["날씨"] = long["날씨"].map({"heat_p": "폭염", "rainy_p": "비", "storm_p": "호우일"})
val = eff.melt(id_vars=["region", "ind_grp"], value_vars=["heat", "rainy", "storm"],
               var_name="날씨", value_name="효과")
val["날씨"] = val["날씨"].map({"heat": "폭염", "rainy": "비", "storm": "호우일"})
long = long.merge(val, on=["region", "ind_grp", "날씨"])
long["q"] = multipletests(long["p"], method="fdr_bh")[1]

sig = long[long["p"] < 0.05].sort_values("q")
sig.to_csv(TAB / "06_J1_fdr.csv", index=False, encoding="utf-8-sig")
print(f"보정 전 유의: {(long['p'] < 0.05).sum()}개 / 보정 후 유의(q<0.05): {(long['q'] < 0.05).sum()}개 / 검정 {len(long)}개")
print(sig.assign(효과=sig["효과"].map("{:+.1%}".format), p=sig["p"].round(4), q=sig["q"].round(4),
                 판정=np.where(sig["q"] < 0.05, "✅ 보정 후 유지", np.where(sig["q"] < 0.10, "△ 경계", "❌ 사라짐")))
      [["region", "ind_grp", "날씨", "효과", "p", "q", "판정"]].to_string(index=False))

# H4 (연령 × 유형) 12개 검정도 보정
e3["q"] = multipletests(e3["heat_p"], method="fdr_bh")[1]
print("\n[H4 연령 × 유형] 폭염 효과 보정")
print(e3.assign(heat=e3["heat"].map("{:+.1%}".format), heat_p=e3["heat_p"].round(4), q=e3["q"].round(4))
      [["region", "type", "age_grp", "heat", "heat_p", "q"]].to_string(index=False))

J2. 연속된 날짜 상관 보정

In [ ]:
t_age = a2.groupby(["date", "region", "type", "age_grp"])["sales"].sum().reset_index()
t_age = add_weather_flags(t_age.merge(days, on=["date", "region"]))

def key_data(region, g):
    return grp[(grp["region"] == region) & (grp["ind_grp"] == g)].sort_values("date")

def age_data(region, typ, age):
    d = t_age[(t_age["region"] == region) & (t_age["type"] == typ) & (t_age["age_grp"] == age)]
    return d.sort_values("date")

KEY_RESULTS = [
    ("① 춘천 야외레저 호우일", key_data("춘천", "야외레저"), "rainy + heavy"),
    ("② 춘천 카페·제과 호우일", key_data("춘천", "카페·제과"), "rainy + heavy"),
    ("② 춘천 의료 호우일", key_data("춘천", "의료"), "rainy + heavy"),
    ("③ 강남 대형유통 폭염", key_data("강남", "대형유통"), "heat"),
    ("④ 춘천 60대+ 여가·외출형 폭염", age_data("춘천", "여가·외출형", "60대 이상"), "heat"),
]
rows = []
for name, d, term in KEY_RESULTS:
    base = smf.ols(FORMULA, data=d)
    for label, fit in [("HC1 (기존)", base.fit(cov_type="HC1")),
                       ("Newey-West 7일", base.fit(cov_type="HAC", cov_kwds={"maxlags": 7}))]:
        tt = fit.t_test(f"{term} = 0")
        rows.append({"결과": name, "표준오차": label,
                     "효과": f"{np.exp(float(tt.effect[0])) - 1:+.1%}",
                     "p": round(float(tt.pvalue), 4)})
nw = pd.DataFrame(rows)
nw.to_csv(TAB / "06_J2_newey_west.csv", index=False, encoding="utf-8-sig")
print(nw.pivot(index="결과", columns="표준오차", values="p").to_string())

J3. 가짜 폭염 검정

In [ ]:
fall_age = full[full["month"].isin([10, 11]) & (full["holiday"] == 0) & (full["chuseok"] == 0)]
fall_days = fall_age[["date", "region", "dow", "week", "tmax", "feels_max", "rain", "heat_streak"]].drop_duplicates()
fa = a2[a2["date"].isin(fall_days["date"])]
fa = fa.groupby(["date", "region", "type", "age_grp"])["sales"].sum().reset_index()
fa = add_weather_flags(fa.merge(fall_days, on=["date", "region"]), heat_c=99)
d = fa[(fa["region"] == "춘천") & (fa["type"] == "여가·외출형") & (fa["age_grp"] == "60대 이상")].reset_index(drop=True)

est = []
for _ in range(200):
    d["heat"] = d["date"].isin(set(rng.choice(d["date"].unique(), 24, replace=False))).astype(int)
    est.append(np.exp(smf.ols(FORMULA, data=d).fit().params["heat"]) - 1)
lo, hi = np.percentile(est, [2.5, 97.5])
print(f"가짜 폭염 평균 {np.mean(est):+.1%}, 95% 범위 {lo:+.1%} ~ {hi:+.1%}, 실제 −18.9% → 범위 밖: {-0.189 < lo}")

## 결과 요약 (2026-10-04 실행 기준)

**근거 수준**
- ✅ **확실**: 다중 검정 보정(J1)·Newey-West(J2)·가짜 폭염 검정(J3)을 통과
- △ **경향**: 여러 분석에서 같은 방향이지만 다중 검정 보정 후 비유의
- 괄호 안 알파벳은 근거가 나온 섹션

### 한 줄 요약
폭우는 춘천의 관광 소비를 끊고, 폭염은 춘천 고령층의 외출을 멈춘다. 같은 날씨에 강남에서는 통계적으로 확인되는 피해가 없으며, 폭염 시 실내 업종으로 소비가 이동하는 경향을 보인다.

### 핵심 결과 1. 춘천 야외레저 × 호우 ✅
- 호우일 −53.4%*, 강남보다 −72.3%p* 더 감소 (A, B)
- 강수량이 많을수록 단계적 감소: 1~10mm −12.5%, 10~30mm −30.5%, 30mm+ −56.2%* (E)
- 시간대와 무관하게 하루 전체가 감소: 06_11 −57.0%*, 12_17 −55.4%*, 18_23 −55.0%* (I1)
- 외지인 −58.5%*, 지역민 −40.9%* (C)
- 폭염·호우 기준을 바꿔도 −51%~−58%로 모두 유의 (G)
- 78개 동시 보정 후 q < 0.001, Newey-West p < 0.001 (J1, J2)

### 핵심 결과 2. 춘천은 호우에 여러 업종이 흔들림, 강남은 버팀 ✅
- 보정 후에도 유의한 춘천 호우 감소: 야외레저, 자동차 −8.0% (q = 0.001), 카페·제과 −8.4% (q = 0.010) (J1)
- 보조 근거: 춘천 의료 −8.9%* (보정 전 유의, Newey-West p = 0.025)
- **강남은 폭염·호우 모두 보정 후 유의한 피해 없음** (J1)
- 춘천 호우 때 외지인 소비 감소: 음식점 −10.8%* (지역민 −1.3%), 카페·제과 −18.2%*, 스포츠 −22.4%* (C)
- 춘천 카페·제과는 10mm부터 유의하게 감소 −8.2%* (E)
- **비가 오는 그 시간대**로 맞추면 하루 단위에서 희석되던 효과가 드러남 (I2)
  - 강남 대형유통(실내): 영향 없음
  - 강남 카페·제과(거리): 약 −6%* (모든 시간대)
  - 춘천 카페·제과: −9%~−17%* (강남의 2~3배)
  - 춘천 음식점: −5%~−9%* (하루 단위로는 비유의였음)
- 춘천 카페(−18.0%*)·음식점(−8.2%*)은 오전 감소가 가장 큼 → 나들이 여부를 아침에 결정 (I1)

### 핵심 결과 3. 강남 폭염 × 실내 이동 △
- 대형유통 +6.2%*, 야외레저(골프장, 강남은 스크린골프·연습장 추정) +17.4%*, 자동차 +6.6%* (A)
- 원업종: 쇼핑몰 +6.1%, 커피전문점 +4.0%*, 서점 +23.2%*, 헬스장 +7.5%*, 일식 +5.5%* (D1)
- 대형유통은 33℃를 넘는 구간에서 +9.7%* (E), 한낮 12_17 +6.4%* (I1)
- 대형유통 외지인 +8.2%* (C), 10~30대 +13.3%* (H3)
- **단, 다중 검정 보정 후 비유의** (대형유통 q = 0.13), 가짜 폭염 범위(−11.8%~+14.5%) 안 (J1, J3)
- → 서로 다른 다섯 가지 분석에서 같은 방향이 반복되므로 "경향"으로 서술

### 핵심 결과 4. 춘천 폭염 × 고령층 외출 감소 ✅
- 춘천 60대 이상 여가·외출형 −18.9%*, 필수형 +2.3% (H4)
- 연령이 높을수록 단계적: +0.7% → −8.5% → −18.9%*
- 강남 60대 이상 여가·외출형은 +24.1% (유의하지 않음) → 같은 고령층, 다른 반응
- 12개 동시 보정 후 q = 0.0003, Newey-West p < 0.001 (J1, J2)
- 가짜 폭염 검정: 우연 범위 −13.7%~+18.7%, 실제 −18.9%는 범위 밖 (J3)

### 신뢰성 점검 요약 (J)
| 결과 | 효과 | 보정 후 q | Newey-West p | 가짜 폭염 | 판정 |
|---|---|---|---|---|---|
| 춘천 야외레저 호우 | −53.4% | < 0.001 | < 0.001 | - | ✅ |
| 춘천 자동차 호우 | −8.0% | 0.001 | - | - | ✅ |
| 춘천 카페·제과 호우 | −8.4% | 0.010 | < 0.001 | - | ✅ |
| 춘천 의료 호우 | −8.9% | 0.21 | 0.025 | - | 보조 |
| 춘천 60대+ 여가 폭염 | −18.9% | 0.0003 | < 0.001 | 범위 밖 | ✅ |
| 강남 대형유통 폭염 | +6.2% | 0.13 | 0.028 | 범위 안 | △ |
| 강남 스크린골프 폭염 | +17.4% | 0.19 | - | - | △ |

- 78개 중 보정 전 유의 14개 → 보정 후 3개 (모두 춘천 호우)
- 가짜 폭염 효과 평균은 모두 0 근처 (+0.2%, +0.1%, −0.0%, +0.9%) → 모형이 없는 효과를 만들어내지 않음

### 쓰지 않을 결과
| 결과 | 이유 |
|---|---|
| 춘천 실내여가 폭염 −9.0%* | 유흥업소(−20.1%)가 주도 (D2). 유흥업소 고령층 −52.1%* (H2). 정책 지원 대상 아님. 보정 후 비유의 |
| 춘천 대형유통 폭염 +37.1%*, 호우 −22.2%* | 정체 불명 기타유통이 주도 |
| 춘천 야외레저 폭염 −10.7%, 저녁 −27.2%* | 31·32℃ 기준에서 부호가 뒤집힘 (G) |
| 춘천 숙박 폭염 −13.9% | 기준 바꾸면 0 근처 (G) |
| 춘천 음식점 폭염 오전 +9.0%* | 업종 그룹 단위에서 비유의, 참고만 (I1) |
| 강남 양식 −14.0%* | 원업종 다중 검정 중 단독 결과 (D1) |
| 임계값 (F) | 신뢰구간이 격자 전체에 걸침 → 뚜렷한 꺾임 없음. 지원 기준은 E 사용 |

### 지원 기준

**언제 (E 기반)**
| 단계 | 조건 | 근거 | 대상 |
|---|---|---|---|
| 호우 주의 | 강수 10mm 이상 예보 | 춘천 카페·제과 −8.2%*, 야외레저 −30.5% | 춘천 관광형 구역 |
| 호우 경보 | 강수 30mm 이상 예보 | 춘천 야외레저 −56.2%* | 춘천 야외레저·관광 업종 |
| 폭염 | 최고기온 33℃ 이상 예보 | 춘천 고령층 여가 −18.9%* ✅, 강남 대형유통 +9.7%* △ | 춘천 고령층 외출·여가 (우선), 강남 골목상권 |

**어떤 시점에 (I 기반)**
| 결과 | 지원 시점·방식 |
|---|---|
| 춘천 야외레저는 하루 전체 손실 | 전날 예보 기준 사전 지원 (예약 취소 보전, 실내 대체 코스) |
| 춘천 카페·음식점은 오전 감소가 가장 큼 | 당일 아침 예보 기반 알림, 오전 프로모션 |
| 비 오는 그 시간대에 매출이 빠짐 | 기상청 단기예보(3시간 단위) 연동 시간대별 알림 |